In [20]:
from pathlib import Path
import spikeinterface.full as si

import numpy as np
import pandas as pd

from kilosort.io import load_ops

data_dir = "/groups/voigts/voigtslab/neuropixels_tests_aug_2024/2024_08_06_npx_long_test/data/output/probe_b/shank_1.0/kilosort/"


# outputs saved to results_dir
results_dir = Path(data_dir).joinpath('phy')
templates =  np.load(results_dir / 'templates.npy')
cluster_groups = pd.read_csv(results_dir / 'cluster_group.tsv', sep = '\t')
cluster_ids = cluster_groups['cluster_id'].values
st = np.load(results_dir / 'spike_times.npy')
clu = np.load(results_dir / 'spike_clusters.npy')

In [21]:
for cluster_group in cluster_ids:
    print(cluster_group)
    n_clu = clu[clu == cluster_group].shape[0]
    print(f"{cluster_group} = {n_clu}")

0
0 = 3
1
1 = 22802
2
2 = 53910
3
3 = 10668
4
4 = 1
5
5 = 102598
6
6 = 59346
7
7 = 169713
8
8 = 278307
9
9 = 214464
10
10 = 452828
11
11 = 381303
12
12 = 400253
13
13 = 1063086
14
14 = 783205
15
15 = 69191
16
16 = 213558
17
17 = 93745
18
18 = 123172
19
19 = 20200
20
20 = 56392
21
21 = 7938
22
22 = 161535
23
23 = 46777
24
24 = 263333
25
25 = 18441
26
26 = 350044
27
27 = 2362
28
28 = 103635
29
29 = 1716
30
30 = 32
31
31 = 210116
32
32 = 66680
33
33 = 137026
34
34 = 116146
35
35 = 321990
36
36 = 22039
37
37 = 101571
38
38 = 330470
39
39 = 196109
40
40 = 65422
41
41 = 43524
42
42 = 303498
43
43 = 180254
44
44 = 216294
45
45 = 60065
46
46 = 168902
47
47 = 30066
48
48 = 129468
49
49 = 15794
50
50 = 33096
51
51 = 265119
52
52 = 413383
53
53 = 320121
54
54 = 11283
55
55 = 16914
56
56 = 77954
57
57 = 76796
58
58 = 92169
59
59 = 111725
60
60 = 197151
61
61 = 205
62
62 = 51
63
63 = 10385
64
64 = 2360
65
65 = 84614
66
66 = 295287
67
67 = 317166
68
68 = 613534
69
69 = 134290
70
70 = 29206
71
71 = 4

In [22]:
from pathlib import Path
import numpy as np
import pandas as pd
import shutil

# Set paths
data_dir = "/groups/voigts/voigtslab/neuropixels_tests_aug_2024/2024_08_06_npx_long_test/data/output/probe_b/shank_1.0/kilosort/"
results_dir = Path(data_dir).joinpath('phy')
MIN_SPIKES = 20

try:
    # Load essential files
    st = np.load(results_dir / 'spike_times.npy')
    clu = np.load(results_dir / 'spike_clusters.npy')
    cluster_groups = pd.read_csv(results_dir / 'cluster_group.tsv', sep='\t')
    
    # Identify sparse clusters
    unique_clusters, spike_counts = np.unique(clu, return_counts=True)
    sparse_clusters = unique_clusters[spike_counts < MIN_SPIKES]
    print(f"Removing {len(sparse_clusters)} clusters with fewer than {MIN_SPIKES} spikes")
    
    # Create mask and remap clusters
    keep_mask = ~np.isin(clu, sparse_clusters)
    remaining_clusters = np.sort(unique_clusters[spike_counts >= MIN_SPIKES])
    new_id_map = {int(old_id): new_id for new_id, old_id in enumerate(remaining_clusters)}
    
    # Backup essential files
    backup_dir = results_dir / 'backup_before_sparsity_filter'
    backup_dir.mkdir(exist_ok=True)
    for file in ['spike_times.npy', 'spike_clusters.npy', 'cluster_group.tsv']:
        shutil.copy(results_dir / file, backup_dir / file)
    
    # Filter and save essential files
    np.save(results_dir / 'spike_times.npy', st[keep_mask])
    np.save(results_dir / 'spike_clusters.npy', 
           np.array([new_id_map[int(c)] for c in clu[keep_mask]]))
    
    # Update cluster groups
    new_cluster_groups = cluster_groups[~cluster_groups['cluster_id'].isin(sparse_clusters)].copy()
    new_cluster_groups['cluster_id'] = new_cluster_groups['cluster_id'].map(lambda x: new_id_map[int(x)])
    new_cluster_groups.to_csv(results_dir / 'cluster_group.tsv', sep='\t', index=False)
    
    print(f"Original spike count: {len(st)}")
    print(f"Filtered spike count: {len(st[keep_mask])}")
    print(f"Files backed up to: {backup_dir}")

except Exception as e:
    print(f"An error occurred: {str(e)}")
    print("No files were modified.")
    raise

Removing 2 clusters with fewer than 20 spikes
Original spike count: 24221821
Filtered spike count: 24221817
Files backed up to: /groups/voigts/voigtslab/neuropixels_tests_aug_2024/2024_08_06_npx_long_test/data/output/probe_b/shank_1.0/kilosort/phy/backup_before_sparsity_filter


In [24]:
import split as sp
import preprocess as pp
import detection as dt
import config as cfg
from pathlib import Path
import utils as ut
import numpy as np


global_probe, global_probe_data = ut.load_probe_from_json(cfg.PROBE_FILE)
probe_names = 'probe_b'
d_fold = "/groups/voigts/voigtslab/neuropixels_tests_aug_2024/2024_08_06_npx_long_test/data/"
recording_files = sp.collect_files(d_fold, probe_names, output_folder=False)
shank_dict, recording_paths = sp.split_recording(recording_files, probe_names, global_probe_data)

num = 1.0
shank = shank_dict['probe_b'][num]
shank_probe = ut.load_probe_from_json(cfg.SHANK_FILE)
recording, artifact_indexes = pp.process_traces(shank, shank_probe, n_channels=cfg.N_CHANNELS_SHANK, num_cpus=12, artifacts=False)

Found 35 files for probe_b
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
7.59
3596.652
0.0
3589.976
3596.813
3596.793
3596.779
3596.857
3441.519
147.997
0.023
3590.177
208.561
3381.037
3596.843
635.746
1148.36
1795.532
0.0
32.031
31.998
3596.738
3596.531
0.0
Split probe_b into 4 shanks
46784.553
Processing phase shift...
Bandpass filtering...
Median removal...


In [27]:
import spikeinterface.exporters as sexp
phy_sorting = si.read_phy(folder_path=str(results_dir))

we = si.create_sorting_analyzer(recording=recording, sorting=phy_sorting, folder=results_dir, overwrite=True, n_jobs=12)
we.compute(['random_spikes', 'waveforms', 'templates', 'noise_levels'], n_jobs=12)
sexp.export_to_phy(we,
                output_folder=str(results_dir + 'new'),
                remove_if_exists=True,
                compute_amplitudes=True,
                compute_pc_features=False,
                copy_binary=False,
                n_jobs=12)

estimate_sparsity:   0%|          | 0/46785 [00:00<?, ?it/s]

create_sorting_analyzer: recording does not have scaling to uV, forcing return_scaled=False


compute_waveforms:   0%|          | 0/46785 [00:00<?, ?it/s]

TypeError: unsupported operand type(s) for +: 'PosixPath' and 'str'

In [28]:
sexp.export_to_phy(we,
                output_folder=str(results_dir / 'new'),
                remove_if_exists=True,
                compute_amplitudes=True,
                compute_pc_features=False,
                copy_binary=False,
                n_jobs=12)

spike_amplitudes:   0%|          | 0/46785 [00:00<?, ?it/s]

Run:
phy template-gui  /groups/voigts/voigtslab/neuropixels_tests_aug_2024/2024_08_06_npx_long_test/data/output/probe_b/shank_1.0/kilosort/phy/new/params.py
